# Stage 2 of 4 — Archive the Official House XML Indexes and Verify PDF Completeness

## What this stage actually does

Stage 1 downloaded the PTR PDFs. Stage 2 asks:

> **Did we actually download every PTR the official House index says should exist?**

For every selected year this notebook:

1. Downloads the official House annual XML financial-disclosure index.
2. Saves the **original XML bytes unchanged** into `Congressional Trading Data/House_PTRs/Original_XML`.
3. Parses the archived XML into a complete filing table.
4. Identifies every PTR (`FilingType == "P"`).
5. Compares those official PTR `DocID` values with the PDFs physically present in `Congressional Trading Data/House_PTRs/<YEAR>/`.
6. Reports missing and extra PDFs.
7. Creates an Excel audit workbook for each year.

## Why you need this stage

A parser can only be complete if the source archive is complete.

Without this verification step, a missing PDF could look exactly like “the politician had no transaction.” Stage 2 gives you an independent completeness check against the House's own annual index and preserves the original indexes for future auditing.

## Outputs

```text
Congressional Trading Data/House_PTRs/
├── 02 Official House XML 03 PDF Archive Verification Reports/
│   ├── 2021FD.xml
│   └── ...
└── 03 PDF Archive Verification Reports/
    ├── House_PTR_Verification_2021.xlsx
    └── ...
```

The Excel workbook also highlights PTR rows:

- green = matching PDF exists in Drive
- red = PTR is listed by the House but its PDF is missing

This stage does **not** alter the source PDFs.

## 1. Mount Google Drive

### Code walkthrough — connect Google Drive

Mount the persistent Drive containing the PDFs downloaded in Stage 1.

In [ ]:
# ============================================================================
# STAGE 2 — GOOGLE DRIVE
# ============================================================================
# Mount the same persistent source archive created by Stage 1.

from google.colab import drive
drive.mount("/content/drive")


## 2. Choose the years

### Code walkthrough — choose verification years

These should normally match the years used by Stage 1.

In [ ]:
# ============================================================================
# STAGE 2 — YEAR RANGE
# ============================================================================
# Choose which official annual House indexes should be archived and checked against local PDFs.

YEARS = [2021, 2022, 2023, 2024, 2025, 2026]


## 3. Download, archive, and verify

### Code walkthrough — archive XML and audit the PDF archive

This cell performs the full verification.

Key idea:

```text
Official House XML PTR DocIDs
            vs.
PDF filenames actually in Drive
```

The difference between those sets identifies missing or unexpected files.

It also stores the original XML before parsing it, so the verification can always be reproduced from the exact official source file used at the time.

In [ ]:
# ============================================================================
# STAGE 2 — OFFICIAL XML ARCHIVE + PDF COMPLETENESS AUDIT
# ============================================================================
# This cell is intentionally one complete verification pass per year.
#
# INPUT EVIDENCE:
# - official House <YEAR>FD.xml index downloaded directly from the Clerk
# - local Congressional Trading Data/House_PTRs/01 Official House PTR PDFs/<YEAR>/*.pdf files created by Stage 1
#
# WHAT IT DOES:
# 1. save the original XML bytes unchanged for reproducibility
# 2. parse every filing record, not only PTRs
# 3. normalize DocID as text
# 4. identify FilingType == "P" as the official PTR set
# 5. collect local PDF stems as the local DocID set
# 6. compare official-vs-local sets for missing and extra PDFs
# 7. write a full Excel audit copy with green/red PTR highlighting
# 8. append one year-level summary record
# 9. display the all-years summary at the end
#
# WHY THIS MATTERS:
# A missing source PDF must never be mistaken for 'no transactions'.
# This audit independently proves whether the Stage 1 source archive matches
# the House's own annual filing index.
#
# RERUN SAFETY:
# The XML and Excel audit files are refreshed. Source PDFs are not changed.

from pathlib import Path
import xml.etree.ElementTree as ET
import requests
import pandas as pd
from openpyxl.styles import PatternFill

BASE_FOLDER = Path("/content/drive/MyDrive/Congressional Trading Data/House_PTRs")
PDF_ARCHIVE_FOLDER = BASE_FOLDER / "01 Official House PTR PDFs"

# Raw official XML files will be saved here.
XML_FOLDER = BASE_FOLDER / "02 Official House XML Indexes"

# Verification Excel files will be saved here.
EXCEL_FOLDER = BASE_FOLDER / "03 PDF Archive Verification Reports"

XML_FOLDER.mkdir(parents=True, exist_ok=True)
EXCEL_FOLDER.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0"
}

summary = []

for year in YEARS:

    print(f"\n===== {year} =====")

    xml_url = (
        f"https://disclosures-clerk.house.gov/"
        f"public_disc/financial-pdfs/{year}FD.xml"
    )

    # Download the official XML.
    response = requests.get(
        xml_url,
        headers=headers,
        timeout=60
    )
    response.raise_for_status()

    # SAVE THE ORIGINAL XML EXACTLY AS DOWNLOADED.
    xml_path = XML_FOLDER / f"{year}FD.xml"
    xml_path.write_bytes(response.content)

    print("Saved original XML:", xml_path)

    # Parse the same saved XML file we just archived.
    root = ET.parse(xml_path).getroot()

    all_rows = []

    for member in root.iter("Member"):

        row = {}

        for child in member:
            row[child.tag] = child.text

        all_rows.append(row)

    # This is the complete official House index, not just PTRs.
    full_index_df = pd.DataFrame(all_rows)

    # Make sure DocID is treated as text.
    if "DocID" in full_index_df.columns:
        full_index_df["DocID"] = (
            full_index_df["DocID"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    # Identify PTR rows.
    ptr_mask = (
        full_index_df["FilingType"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("P")
    )

    ptr_df = full_index_df.loc[ptr_mask].copy()

    # PDFs actually present in your Drive folder.
    year_folder = PDF_ARCHIVE_FOLDER / str(year)
    year_folder.mkdir(exist_ok=True)

    pdf_docids = {
        pdf.stem
        for pdf in year_folder.glob("*.pdf")
    }

    # Add our verification column to the copied spreadsheet data.
    full_index_df["PDF in Drive?"] = ""

    for row_number in full_index_df.index:

        filing_type = str(
            full_index_df.loc[row_number, "FilingType"]
        ).strip()

        doc_id = str(
            full_index_df.loc[row_number, "DocID"]
        ).strip()

        if filing_type == "P":
            full_index_df.loc[
                row_number,
                "PDF in Drive?"
            ] = "YES" if doc_id in pdf_docids else "NO"

    expected_docids = set(
        ptr_df["DocID"].astype(str)
    )

    missing_docids = sorted(
        expected_docids - pdf_docids
    )

    extra_docids = sorted(
        pdf_docids - expected_docids
    )

    expected_count = len(expected_docids)
    pdf_count = len(pdf_docids)
    matched_count = len(expected_docids & pdf_docids)
    missing_count = len(missing_docids)
    extra_count = len(extra_docids)

    complete = (
        missing_count == 0
        and extra_count == 0
    )

    verification_df = pd.DataFrame({
        "Check": [
            "Year",
            "PTRs in official House XML",
            "PDF files in Drive",
            "Matched PTR PDFs",
            "Missing PTR PDFs",
            "Extra PDFs not in XML",
            "Complete?"
        ],
        "Result": [
            year,
            expected_count,
            pdf_count,
            matched_count,
            missing_count,
            extra_count,
            "YES" if complete else "NO"
        ]
    })

    missing_df = pd.DataFrame({
        "Missing DocID": missing_docids
    })

    extra_df = pd.DataFrame({
        "Extra PDF DocID": extra_docids
    })

    excel_path = (
        EXCEL_FOLDER /
        f"House_PTR_Verification_{year}.xlsx"
    )

    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl"
    ) as writer:

        # Entire original House index plus one added verification column.
        full_index_df.to_excel(
            writer,
            sheet_name="House XML Index",
            index=False
        )

        verification_df.to_excel(
            writer,
            sheet_name="Verification",
            index=False
        )

        if missing_count:
            missing_df.to_excel(
                writer,
                sheet_name="Missing PTRs",
                index=False
            )

        if extra_count:
            extra_df.to_excel(
                writer,
                sheet_name="Extra PDFs",
                index=False
            )

        # Highlight PTR rows in the House XML Index sheet.
        worksheet = writer.book["House XML Index"]

        headers_in_sheet = {
            cell.value: cell.column
            for cell in worksheet[1]
        }

        filing_col = headers_in_sheet["FilingType"]
        drive_col = headers_in_sheet["PDF in Drive?"]

        green_fill = PatternFill(
            fill_type="solid",
            fgColor="C6EFCE"
        )

        red_fill = PatternFill(
            fill_type="solid",
            fgColor="FFC7CE"
        )

        # Row 1 is headers, so worksheet row 2 corresponds
        # to DataFrame row 0.
        for excel_row in range(
            2,
            worksheet.max_row + 1
        ):

            filing_type = worksheet.cell(
                excel_row,
                filing_col
            ).value

            drive_status = worksheet.cell(
                excel_row,
                drive_col
            ).value

            # Only highlight PTR rows.
            if filing_type == "P":

                fill = (
                    green_fill
                    if drive_status == "YES"
                    else red_fill
                )

                for cell in worksheet[excel_row]:
                    cell.fill = fill

    summary.append({
        "Year": year,
        "House PTRs": expected_count,
        "PDFs in Drive": pdf_count,
        "Matched": matched_count,
        "Missing": missing_count,
        "Extra": extra_count,
        "Complete": complete
    })

    print("House PTRs:", expected_count)
    print("PDFs in Drive:", pdf_count)
    print("Matched:", matched_count)
    print("Missing:", missing_count)
    print("Extra:", extra_count)
    print("Complete:", complete)
    print("Excel:", excel_path)


summary_df = pd.DataFrame(summary)

print("\n===== ALL YEARS =====")
display(summary_df)


## Files created in Google Drive

After Stage 2, your archive will look like:

```text
My Drive
└── Congressional Trading Data/House_PTRs
    ├── 01 Official House PTR PDFs
    │   ├── 2021
    │   ├── 2022
    │   ├── 2023
    │   ├── 2024
    │   ├── 2025
    │   └── 2026
    │
    ├── 02 Official House XML Indexes
    │   ├── 2021FD.xml
    │   ├── 2022FD.xml
    │   ├── 2023FD.xml
    │   ├── 2024FD.xml
    │   ├── 2025FD.xml
    │   └── 2026FD.xml
    │
    └── 03 PDF Archive Verification Reports
        ├── House_PTR_Verification_2021.xlsx
        ├── House_PTR_Verification_2022.xlsx
        ├── House_PTR_Verification_2023.xlsx
        ├── House_PTR_Verification_2024.xlsx
        ├── House_PTR_Verification_2025.xlsx
        └── House_PTR_Verification_2026.xlsx
```